# 🚦 Pipeline Xử Lý Dữ Liệu CNN - Phân Loại Biển Báo Giao Thông Việt Nam

**Mục tiêu:** Xây dựng pipeline xử lý dữ liệu hoàn chỉnh cho bài toán Image Classification sử dụng CNN.

---

## 📋 Mục lục Pipeline:

| # | Bước | Mô tả |
|---|------|-------|
| 1 | **Setup & Configuration** | Import thư viện và thiết lập cấu hình |
| 2 | **Helper Functions** | Các hàm xử lý cốt lõi (Resize, Dedup, Augment) |
| 3 | **Data Splitting** | Gộp và chia dữ liệu theo tỉ lệ 70/20/10 |
| 4 | **EDA - Trước xử lý** | Phân tích phân phối dữ liệu ban đầu |
| 5 | **Main Processing Loop** | Dedup → Resize → Balance |
| 6 | **EDA - Sau xử lý** | Kiểm tra kết quả |
| 7 | **Export & Upload Kaggle** | Đóng gói và upload |

---

## 🎯 Đặc điểm kỹ thuật:

| Kỹ thuật | Mô tả | Lý do |
|----------|-------|-------|
| **Resize with Padding** | Giữ nguyên tỉ lệ ảnh, thêm viền đen | Tránh méo biển báo (tròn, tam giác) |
| **No Horizontal Flip** | Không lật ngang | Biển báo có hướng cố định (mũi tên, chữ số) |
| **Auto-Balancing** | Augment để cân bằng số lượng mẫu | Tránh model bias về class đông |
| **Deduplication** | Loại bỏ ảnh trùng bằng pHash | Tránh overfit do ảnh trùng lặp |
| **Auto-Splitting** | Tự động chia 70/20/10 | Đảm bảo tỉ lệ train/val/test hợp lý |

---

## 📦 Thư viện cần thiết:
```bash
pip install opencv-python numpy pillow matplotlib seaborn tqdm albumentations imagehash
```

## 1. 🛠️ Setup & Configuration

Import các thư viện cần thiết và thiết lập cấu hình cho pipeline.

In [ ]:
# =============================================================================
# CELL 1: IMPORT LIBRARIES & CONFIGURATION
# =============================================================================
# Mục đích: Import tất cả thư viện cần thiết và thiết lập cấu hình tập trung
# =============================================================================

import os
import sys
import shutil
import random
import hashlib
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict, Optional

# =============================================================================
# SETUP PATHS - Thiết lập đường dẫn dự án
# =============================================================================
# Notebook này nằm trong: Traffic_Sign_Project/Data_Pipeline/
# Cần import Utils từ: Traffic_Sign_Project/Utils/

NOTEBOOK_DIR = Path.cwd()  # Thư mục hiện tại khi chạy notebook
PROJECT_ROOT = NOTEBOOK_DIR.parent  # Traffic_Sign_Project/
PARENT_DIR = PROJECT_ROOT.parent    # Project_DL/ (chứa Data/, output_*)

# Thêm Utils vào Python path
sys.path.insert(0, str(PROJECT_ROOT))

# Import helper functions
try:
    from Utils.helper_functions import (
        resize_with_padding,
        TRAFFIC_SIGN_CLASSES,
        ensure_dir,
    )
    print("✅ Imported helper functions from Utils")
except ImportError as e:
    print(f"⚠️ Could not import helper functions: {e}")
    print("   Sử dụng cấu hình local...")

# Progress bar
from tqdm import tqdm

# Image processing
import cv2
import numpy as np
from PIL import Image

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Augmentation - Thư viện mạnh mẽ cho image augmentation
try:
    import albumentations as A
    ALBUMENTATIONS_AVAILABLE = True
except ImportError:
    print("⚠️ albumentations chưa được cài đặt. Chạy: pip install albumentations")
    ALBUMENTATIONS_AVAILABLE = False

# Perceptual Hashing - Phát hiện ảnh trùng lặp
try:
    import imagehash
    IMAGEHASH_AVAILABLE = True
except ImportError:
    print("⚠️ imagehash chưa được cài đặt. Chạy: pip install imagehash")
    IMAGEHASH_AVAILABLE = False

# =============================================================================
# VISUALIZATION SETTINGS
# =============================================================================
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Seed cho reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# =============================================================================
# CONFIGURATION - TẬP TRUNG TẤT CẢ CẤU HÌNH Ở ĐÂY
# =============================================================================
# Đường dẫn được tính relative từ thư mục cha (Project_DL/)
CONFIG = {
    # === Đường dẫn - Relative từ PARENT_DIR (Project_DL/) ===
    'input_dir': str(PARENT_DIR / 'Data' / 'cnn_dataset'),
    'output_dir': str(PARENT_DIR / 'output_CNNdataset'),
    
    # === Kích thước ảnh đầu ra ===
    # 64x64: nhẹ, train nhanh | 128x128: cân bằng | 224x224: chính xác cao
    'target_size': (64, 64),
    
    # === Tỉ lệ chia dữ liệu (Train / Val / Test) ===
    'split_ratios': (0.7, 0.2, 0.1),      # 70% train, 20% val, 10% test
    
    # === Augmentation settings ===
    'balance_data': True,                  # Tự động cân bằng số lượng mẫu
    'augment_train_only': True,            # Chỉ augment tập train
    
    # === Deduplication ===
    'phash_threshold': 5,                  # Ngưỡng Hamming distance (<=5 = gần giống)
    
}

# =============================================================================
# TẠO CÁC ĐƯỜNG DẪN TỪ CONFIG
# =============================================================================
INPUT_DIR = Path(CONFIG['input_dir'])
OUTPUT_DIR = Path(CONFIG['output_dir'])
TARGET_SIZE = CONFIG['target_size']
SPLIT_RATIOS = CONFIG['split_ratios']

# =============================================================================
# IN THÔNG TIN CẤU HÌNH
# =============================================================================
print("=" * 60)
print("✅ LIBRARIES IMPORTED SUCCESSFULLY")
print("=" * 60)
print(f"\n📁 Project Structure:")
print(f"   PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   PARENT_DIR:   {PARENT_DIR}")
print(f"\n📁 Data Paths:")
print(f"   Input Directory:  {INPUT_DIR}")
print(f"   Output Directory: {OUTPUT_DIR}")
print(f"\n📐 Target Size:      {TARGET_SIZE[0]}×{TARGET_SIZE[1]}")
print(f"📊 Split Ratios:     {int(SPLIT_RATIOS[0]*100)}% / {int(SPLIT_RATIOS[1]*100)}% / {int(SPLIT_RATIOS[2]*100)}% (train/val/test)")
print(f"\n🔧 Albumentations:   {'✅ Available' if ALBUMENTATIONS_AVAILABLE else '❌ Not installed'}")
print(f"🔧 ImageHash:        {'✅ Available' if IMAGEHASH_AVAILABLE else '❌ Not installed'}")

In [ ]:
# =============================================================================
# CELL 2: SETUP PATHS & DIRECTORIES
# =============================================================================
# Sử dụng helper functions để lấy đường dẫn chuẩn
# =============================================================================

from pathlib import Path

# Import path functions từ Utils
try:
    from Utils.helper_functions import (
        get_paths, 
        get_cnn_picture_paths, 
        setup_directories,
        save_figure,
        save_image,
        save_comparison_image,
        resize_with_padding,
    )
    PATHS = get_paths()
    CNN_PICS = get_cnn_picture_paths()
    USE_NEW_PATHS = True
    print("✅ Imported path functions from Utils")
except ImportError:
    USE_NEW_PATHS = False
    print("⚠️ Using legacy paths")

# =============================================================================
# ĐƯỜNG DẪN DỮ LIỆU
# =============================================================================
if USE_NEW_PATHS:
    # Đường dẫn mới (cấu trúc chuẩn)
    INPUT_DIR = PATHS['cnn_raw_data']
    OUTPUT_DIR = PATHS['cnn_processed']
    PICTURES_DIR = PATHS['cnn_pictures']
    
    # Fallback nếu thư mục mới chưa có data
    if not INPUT_DIR.exists():
        INPUT_DIR = PATHS['cnn_raw_data_legacy']
        print(f"⚠️ Using legacy input: {INPUT_DIR}")
else:
    # Đường dẫn cũ (fallback)
    INPUT_DIR = Path(CONFIG['input_dir'])
    OUTPUT_DIR = Path(CONFIG['output_dir'])
    PICTURES_DIR = PARENT_DIR / 'Pictures' / 'CNN_Processing'

# =============================================================================
# CÁC THƯ MỤC LƯU ẢNH VISUALIZATION
# =============================================================================
if USE_NEW_PATHS:
    PIC_RAW_SAMPLES = CNN_PICS['01_raw_samples']
    PIC_CLASS_DIST = CNN_PICS['02_class_distribution']
    PIC_RESIZE = CNN_PICS['03_resize_padding']
    PIC_DUPLICATE = CNN_PICS['04_duplicate_detection']
    PIC_AUGMENT = CNN_PICS['05_augmentation']
    PIC_FINAL_DIST = CNN_PICS['06_final_distribution']
    PIC_SAMPLES = CNN_PICS['07_sample_per_class']
else:
    PIC_RAW_SAMPLES = PICTURES_DIR / '01_Raw_Samples'
    PIC_CLASS_DIST = PICTURES_DIR / '02_Class_Distribution'
    PIC_RESIZE = PICTURES_DIR / '03_Resize_Padding'
    PIC_DUPLICATE = PICTURES_DIR / '04_Duplicate_Detection'
    PIC_AUGMENT = PICTURES_DIR / '05_Augmentation'
    PIC_FINAL_DIST = PICTURES_DIR / '06_Final_Distribution'
    PIC_SAMPLES = PICTURES_DIR / '07_Sample_Per_Class'

# =============================================================================
# TẠO TẤT CẢ THƯ MỤC
# =============================================================================
# Tạo thư mục output với cấu trúc train/val/test và 7 classes
CLASS_NAMES = [
    '0_Cam_nguoc_chieu', '1_Cam_dung_va_do', '2_Cam_re',
    '3_Gioi_han_toc_do', '4_Cam_con_lai', '5_Nguy_hiem', '6_Hieu_lenh'
]

all_dirs = [
    OUTPUT_DIR, PICTURES_DIR,
    PIC_RAW_SAMPLES, PIC_CLASS_DIST, PIC_RESIZE, PIC_DUPLICATE,
    PIC_AUGMENT, PIC_FINAL_DIST, PIC_SAMPLES
]

# Tạo thư mục cho từng class trong train/val/test
for split in ['train', 'val', 'test']:
    for class_name in CLASS_NAMES:
        all_dirs.append(OUTPUT_DIR / split / class_name)

for d in all_dirs:
    d.mkdir(parents=True, exist_ok=True)

# =============================================================================
# IN THÔNG TIN ĐƯỜNG DẪN
# =============================================================================
print("=" * 60)
print("📁 DATA PATHS")
print("=" * 60)
print(f"  📥 INPUT_DIR:  {INPUT_DIR}")
print(f"  📤 OUTPUT_DIR: {OUTPUT_DIR}")
print(f"  🖼️ PICTURES:   {PICTURES_DIR}")
print()
print("📂 Output structure:")
print(f"  • train/ (7 classes)")
print(f"  • val/   (7 classes)")
print(f"  • test/  (7 classes)")
print()
print("🖼️ Picture directories:")
print(f"  • 01_Raw_Samples")
print(f"  • 02_Class_Distribution")
print(f"  • 03_Resize_Padding")
print(f"  • 04_Duplicate_Detection")
print(f"  • 05_Augmentation")
print(f"  • 06_Final_Distribution")
print(f"  • 07_Sample_Per_Class")

## 2. 🔧 Helper Functions

Các hàm xử lý cốt lõi cho pipeline:
- **`resize_with_padding`**: Resize giữ nguyên tỷ lệ, thêm viền đen
- **`find_duplicates`**: Phát hiện ảnh trùng lặp bằng pHash
- **`augment_image`**: Pipeline augmentation cho traffic signs
- **`get_class_distribution`**: Đếm số lượng ảnh mỗi class

In [ ]:
# =============================================================================
# CELL 2: HELPER FUNCTIONS - CÁC HÀM XỬ LÝ CỐT LÕI
# =============================================================================

def resize_with_padding(image: np.ndarray, target_size: Tuple[int, int]) -> np.ndarray:
    """
    Resize ảnh về kích thước target nhưng GIỮ NGUYÊN TỈ LỆ (Aspect Ratio).
    
    TẠI SAO CẦN PADDING?
    - Biển báo giao thông có hình dạng đặc trưng (tròn, tam giác, chữ nhật)
    - Nếu resize trực tiếp sẽ làm MÉMO hình dạng → Model học sai feature
    - Padding giữ nguyên tỉ lệ, thêm viền đen để đạt kích thước mong muốn
    
    Thuật toán:
    1. Tính tỉ lệ scale để fit vào target_size
    2. Resize ảnh theo tỉ lệ đó
    3. Thêm padding (viền đen) để đạt đúng target_size
    
    Args:
        image: Ảnh đầu vào (numpy array, BGR format)
        target_size: Tuple (width, height) mong muốn
        
    Returns:
        Ảnh đã resize với padding (numpy array)
    """
    target_w, target_h = target_size
    h, w = image.shape[:2]
    
    # Tính tỉ lệ scale để fit vào target, giữ nguyên aspect ratio
    scale = min(target_w / w, target_h / h)
    
    # Kích thước mới sau khi scale
    new_w = int(w * scale)
    new_h = int(h * scale)
    
    # Resize ảnh
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # Tạo canvas đen với kích thước target
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    
    # Tính vị trí để đặt ảnh ở giữa canvas
    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2
    
    # Đặt ảnh vào giữa canvas
    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    
    return canvas


def compute_phash(image_path: str) -> Optional[str]:
    """
    Tính Perceptual Hash cho một ảnh.
    
    pHash hoạt động bằng cách:
    1. Resize ảnh về kích thước nhỏ (32x32)
    2. Chuyển sang grayscale
    3. Áp dụng DCT (Discrete Cosine Transform)
    4. Lấy các hệ số low-frequency làm hash
    
    Args:
        image_path: Đường dẫn tới file ảnh
        
    Returns:
        Hash string hoặc None nếu lỗi
    """
    if not IMAGEHASH_AVAILABLE:
        return None
    
    try:
        img = Image.open(image_path)
        return str(imagehash.phash(img))
    except Exception:
        return None


def find_duplicates(image_paths: List[str], threshold: int = 5) -> List[str]:
    """
    Tìm và trả về danh sách các ảnh trùng lặp cần loại bỏ.
    
    TẠI SAO CẦN DEDUPLICATION?
    - Dataset có thể chứa nhiều ảnh gần giống nhau (từ video, copy)
    - Ảnh trùng gây overfit, model "nhớ" thay vì "học"
    - Loại bỏ duplicate giúp dataset đa dạng hơn
    
    Thuật toán:
    1. Tính pHash cho tất cả ảnh
    2. So sánh pair-wise bằng Hamming distance
    3. Gom nhóm các ảnh giống nhau (distance <= threshold)
    4. Giữ lại 1 ảnh đại diện, loại bỏ còn lại
    
    Args:
        image_paths: Danh sách đường dẫn ảnh
        threshold: Ngưỡng Hamming distance (mặc định 5)
        
    Returns:
        Danh sách đường dẫn ảnh cần loại bỏ (duplicates)
    """
    if not IMAGEHASH_AVAILABLE or len(image_paths) < 2:
        return []
    
    # Tính hash cho tất cả ảnh
    hashes = {}
    for path in image_paths:
        h = compute_phash(path)
        if h:
            hashes[path] = imagehash.hex_to_hash(h)
    
    if len(hashes) < 2:
        return []
    
    # Tìm duplicates bằng Union-Find
    parent = {p: p for p in hashes.keys()}
    
    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    
    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py
    
    paths = list(hashes.keys())
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            distance = hashes[paths[i]] - hashes[paths[j]]
            if distance <= threshold:
                union(paths[i], paths[j])
    
    # Gom nhóm
    groups = {}
    for p in paths:
        root = find(p)
        if root not in groups:
            groups[root] = []
        groups[root].append(p)
    
    # Giữ lại 1 ảnh mỗi nhóm, loại bỏ còn lại
    duplicates_to_remove = []
    for group in groups.values():
        if len(group) > 1:
            # Sắp xếp và giữ ảnh đầu tiên
            sorted_group = sorted(group)
            duplicates_to_remove.extend(sorted_group[1:])
    
    return duplicates_to_remove


def create_augmentation_pipeline():
    """
    Tạo pipeline augmentation cho biển báo giao thông.
    
    NGUYÊN TẮC AUGMENTATION CHO TRAFFIC SIGNS:
    
    ❌ KHÔNG DÙNG:
    - HorizontalFlip: Biển báo có hướng cố định (mũi tên, chữ số)
    - VerticalFlip: Biển báo luôn thẳng đứng
    - Rotation > 15°: Biển báo thường thẳng, không nghiêng nhiều
    
    ✅ NÊN DÙNG:
    - Rotation nhẹ (±15°): Mô phỏng góc camera
    - Brightness/Contrast: Mô phỏng điều kiện ánh sáng
    - MotionBlur: Mô phỏng camera di chuyển
    - GaussianNoise: Mô phỏng nhiễu sensor
    - ColorJitter: Mô phỏng điều kiện thời tiết
    
    Returns:
        Albumentations Compose pipeline
    """
    if not ALBUMENTATIONS_AVAILABLE:
        return None
    
    transform = A.Compose([
        A.Rotate(limit=10, p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.MotionBlur(p=0.3),
        A.GaussNoise(p=0.3),
        A.HueSaturationValue(p=0.3),
        A.GaussianBlur(p=0.3)
    ])
    
    return transform


def augment_image(image: np.ndarray, transform) -> np.ndarray:
    """
    Áp dụng augmentation lên một ảnh.
    
    Args:
        image: Ảnh đầu vào (numpy array, BGR format)
        transform: Albumentations transform pipeline
        
    Returns:
        Ảnh sau khi augment (numpy array)
    """
    if transform is None:
        return image
    
    # Albumentations yêu cầu RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    augmented = transform(image=image_rgb)['image']
    # Chuyển lại BGR để lưu với cv2
    return cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)


def get_class_distribution(data_dir: Path) -> Dict[str, int]:
    """
    Đếm số lượng ảnh trong mỗi class.
    
    Args:
        data_dir: Đường dẫn tới thư mục chứa các folder class
        
    Returns:
        Dictionary {class_name: count}
    """
    distribution = {}
    
    if not data_dir.exists():
        return distribution
    
    for class_dir in sorted(data_dir.iterdir()):
        if class_dir.is_dir():
            # Đếm các file ảnh
            images = list(class_dir.glob('*.jpg')) + \
                     list(class_dir.glob('*.jpeg')) + \
                     list(class_dir.glob('*.png'))
            distribution[class_dir.name] = len(images)
    
    return distribution


def load_image_safe(image_path: str) -> Optional[np.ndarray]:
    """
    Load ảnh với xử lý ngoại lệ.
    
    Args:
        image_path: Đường dẫn tới file ảnh
        
    Returns:
        Numpy array hoặc None nếu lỗi
    """
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        return img
    except Exception:
        return None


print("✅ Helper functions defined successfully!")
print("   - resize_with_padding(): Resize giữ tỉ lệ với padding")
print("   - find_duplicates(): Tìm ảnh trùng lặp bằng pHash")
print("   - create_augmentation_pipeline(): Tạo pipeline augmentation")
print("   - augment_image(): Áp dụng augmentation")
print("   - get_class_distribution(): Đếm số ảnh mỗi class")

## 2.5. 🔀 Data Splitting (Gộp & Chia Dữ Liệu)

**Mục đích:** Tự động chia dataset thành 3 tập **Train / Val / Test** theo tỉ lệ cấu hình.

### Quy trình:
1. Kiểm tra xem `INPUT_DIR` đã có sẵn các folder `train/`, `val/`, `test/` chưa
2. Nếu **chưa có** → Gộp tất cả ảnh theo class và chia theo tỉ lệ **70/20/10**
3. Nếu **đã có** → Sử dụng trực tiếp, không chia lại

### Tham số:
- `ratios`: Tuple tỉ lệ (train, val, test) — mặc định `(0.7, 0.2, 0.1)`
- `seed`: Random seed để đảm bảo reproducibility
- `move_files`: `False` = copy (giữ nguyên gốc), `True` = di chuyển

In [ ]:
# =============================================================================
# CELL: MERGE & SPLIT DATA INTO TRAIN/VAL/TEST
# =============================================================================
# Mục đích: Gộp và chia dữ liệu theo tỉ lệ từ CONFIG['split_ratios']
# Mặc định: 70% train / 20% val / 10% test
# =============================================================================

def split_dataset_into_splits(
    source_dir: Path, 
    output_dir: Path, 
    ratios: Tuple[float, float, float] = (0.7, 0.2, 0.1), 
    seed: int = RANDOM_SEED, 
    move_files: bool = False
) -> Optional[Path]:
    """
    Gộp tất cả ảnh theo từng class và chia thành 3 tập: train/val/test.

    Xử lý 2 trường hợp:
    1. Source có sẵn train/val/test → sử dụng trực tiếp
    2. Source có train/val (không có test) hoặc chỉ có class folders 
       → gộp lại và chia theo ratios mới

    Args:
        source_dir: Thư mục chứa dữ liệu (có thể là class folders hoặc train/val/test)
        output_dir: Thư mục đầu ra chứa train/val/test (we now write splits directly into OUTPUT_DIR)
        ratios: Tuple tỉ lệ (train, val, test) — tổng = 1.0
        seed: Random seed để reproducible
        move_files: False = copy, True = di chuyển (xóa gốc)

    Returns:
        Path tới folder chứa splits, hoặc None nếu lỗi
    """
    # Kiểm tra source tồn tại
    if not source_dir.exists():
        print(f"❌ Source dir not found: {source_dir}")
        return None

    # Nếu đã có đầy đủ train/val/test → sử dụng luôn
    if all((source_dir / s).exists() for s in ['train', 'val', 'test']):
        print(f"✅ Source đã có sẵn train/val/test splits")
        print(f"   → Sử dụng trực tiếp: {source_dir}")
        return source_dir

    # Validate ratios
    if abs(sum(ratios) - 1.0) > 0.01:
        print(f"⚠️ Tổng ratios = {sum(ratios):.2f} ≠ 1.0, tự động chuẩn hóa...")
        total = sum(ratios)
        ratios = tuple(r / total for r in ratios)  # type: ignore

    # Chuẩn bị output_dir
    # If output_dir is the global OUTPUT_DIR, avoid deleting the entire folder to preserve other files.
    if output_dir.exists():
        if Path(output_dir) == Path(OUTPUT_DIR):
            print(f"⚠️ Output dir exists ({OUTPUT_DIR}), removing existing split subfolders only.")
            for s in ['train', 'val', 'test']:
                p = output_dir / s
                if p.exists():
                    print(f"   - Xóa cũ: {p}")
                    shutil.rmtree(p)
        else:
            print(f"⚠️ Xóa split output cũ: {output_dir}")
            shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Định nghĩa các extension ảnh hỗ trợ
    img_exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')

    # ==========================================================================
    # XÁC ĐỊNH CẤU TRÚC SOURCE: có train/val hay chỉ có class folders?
    # ==========================================================================
    has_train_val = (source_dir / 'train').exists() and (source_dir / 'val').exists()

    if has_train_val:
        print(f"\n📂 Phát hiện cấu trúc train/val có sẵn → Gộp lại và chia theo {int(ratios[0]*100)}/{int(ratios[1]*100)}/{int(ratios[2]*100)}")
        
        # Gom tất cả ảnh từ train + val theo class
        class_images = {}  # {class_name: [list of image paths]}
        
        for split_name in ['train', 'val', 'test']:
            split_dir = source_dir / split_name
            if not split_dir.exists():
                continue
            
            for class_dir in split_dir.iterdir():
                if class_dir.is_dir():
                    class_name = class_dir.name
                    if class_name not in class_images:
                        class_images[class_name] = []
                    
                    # Gom ảnh từ class này
                    for ext in img_exts:
                        class_images[class_name].extend(list(class_dir.glob(ext)))
        
        print(f"   Tìm thấy {len(class_images)} classes")
        total_before = sum(len(v) for v in class_images.values())
        print(f"   Tổng số ảnh: {total_before:,}")
        
    else:
        print(f"\n📂 Cấu trúc class folders → Chia theo {int(ratios[0]*100)}/{int(ratios[1]*100)}/{int(ratios[2]*100)}")
        
        # Gom ảnh từ các class folders trực tiếp
        class_images = {}
        
        for class_dir in source_dir.iterdir():
            if class_dir.is_dir():
                class_name = class_dir.name
                class_images[class_name] = []
                
                for ext in img_exts:
                    class_images[class_name].extend(list(class_dir.glob(ext)))
        
        print(f"   Tìm thấy {len(class_images)} classes")
        total_before = sum(len(v) for v in class_images.values())
        print(f"   Tổng số ảnh: {total_before:,}")
    
    # ==========================================================================
    # CHIA DỮ LIỆU THEO RATIOS
    # ==========================================================================
    total_stats = {}
    
    print(f"\n🔀 Đang chia {len(class_images)} classes...")
    
    for class_name, imgs in tqdm(class_images.items(), desc="Splitting classes"):
        if not imgs:
            continue
        
        # Shuffle với seed cố định để reproducible
        rnd = random.Random(seed)
        rnd.shuffle(imgs)
        
        # Tính số lượng cho mỗi split
        n = len(imgs)
        train_n = int(n * ratios[0])
        val_n = int(n * ratios[1])
        # test_n = phần còn lại để đảm bảo không mất ảnh
        
        splits = {
            'train': imgs[:train_n],
            'val': imgs[train_n:train_n + val_n],
            'test': imgs[train_n + val_n:],
        }
        
        # Copy/Move files vào các folder tương ứng
        for split_name, files in splits.items():
            dst = output_dir / split_name / class_name
            dst.mkdir(parents=True, exist_ok=True)
            
            for fp in files:
                dst_path = dst / fp.name
                if move_files:
                    shutil.move(str(fp), str(dst_path))
                else:
                    shutil.copy2(str(fp), str(dst_path))
        
        total_stats[class_name] = {
            'total': n,
            'train': len(splits['train']),
            'val': len(splits['val']),
            'test': len(splits['test'])
        }
    
    # In tóm tắt
    totals = {'train': 0, 'val': 0, 'test': 0}
    for v in total_stats.values():
        totals['train'] += v['train']
        totals['val'] += v['val']
        totals['test'] += v['test']
    
    print(f"\n✅ Split hoàn tất!")
    print(f"   📁 Output: {output_dir}")
    print(f"   📊 Classes: {len(total_stats)}")
    print(f"   🔢 Images: train={totals['train']:,} | val={totals['val']:,} | test={totals['test']:,}")
    
    return output_dir


# =============================================================================
# THỰC THI SPLIT
# =============================================================================
print("=" * 60)
print("🔀 DATA SPLITTING")
print("=" * 60)

# Xác định thư mục đầu ra cho splits (ghi trực tiếp vào OUTPUT_DIR)
SPLIT_DIR = OUTPUT_DIR

# Thực hiện split (hoặc sử dụng splits có sẵn)
result = split_dataset_into_splits(
    source_dir=INPUT_DIR,
    output_dir=SPLIT_DIR,
    ratios=SPLIT_RATIOS,
    seed=RANDOM_SEED,
    move_files=False  # Copy để giữ nguyên dữ liệu gốc
)

# Xác định thư mục input thực sự cho các bước tiếp theo
if result is not None:
    actual_input_dir = result
else:
    actual_input_dir = INPUT_DIR
    print("⚠️ Fallback về INPUT_DIR")

print(f"\n📁 Sử dụng cho các bước tiếp theo: {actual_input_dir}")


## 3. 📊 EDA - Phân Tích Dữ Liệu TRƯỚC Xử Lý

**Mục đích:** Phân tích phân phối dữ liệu ban đầu để:
- Xác định số lượng class và tên class
- Phát hiện mất cân bằng dữ liệu (Imbalance)
- Quyết định chiến lược augmentation phù hợp

> **Lưu ý:** Cell này sử dụng `actual_input_dir` được set từ bước Split ở trên.

In [ ]:
# =============================================================================
# CELL: EDA - PHÂN TÍCH DỮ LIỆU TRƯỚC XỬ LÝ
# =============================================================================

def plot_class_distribution(distribution: Dict[str, int], title: str, 
                            color: str = 'steelblue', save_path: Path = None):
    """
    Vẽ biểu đồ cột phân phối số lượng ảnh theo class.
    
    Args:
        distribution: Dictionary {class_name: count}
        title: Tiêu đề biểu đồ
        color: Màu của bars
        save_path: Đường dẫn lưu hình (nếu None thì không lưu)
    
    Returns:
        Figure object
    """
    if not distribution:
        print("⚠️ Không có dữ liệu để hiển thị!")
        return None
    
    # Sắp xếp theo tên class
    sorted_items = sorted(distribution.items())
    class_names = [item[0] for item in sorted_items]
    counts = [item[1] for item in sorted_items]
    
    # Tạo figure
    fig, ax = plt.subplots(figsize=(16, 6))
    
    # Vẽ bar chart
    bars = ax.bar(range(len(class_names)), counts, color=color, edgecolor='black', alpha=0.8)
    
    # Thêm giá trị lên mỗi bar
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{count}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    # Cấu hình trục
    ax.set_xlabel('Class', fontsize=12)
    ax.set_ylabel('Number of Images', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    
    # Thêm đường trung bình
    avg = float(np.mean(counts))
    ax.axhline(y=avg, color='red', linestyle='--', linewidth=2, label=f'Average: {avg:.0f}')
    ax.legend(fontsize=10)
    
    plt.tight_layout()
    
    # Lưu hình nếu có save_path
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        # Prefer helper save_figure if available
        if 'save_figure' in globals() and callable(save_figure):
            try:
                save_figure(fig, save_path.name, save_path.parent)
            except Exception:
                fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        else:
            fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"💾 Saved: {save_path}")
    
    plt.show()
    return fig


def analyze_imbalance(distribution: Dict[str, int]) -> Dict:
    """
    Phân tích mức độ mất cân bằng dữ liệu.
    
    Returns:
        Dictionary với các thống kê: total, min, max, mean, std, imbalance_ratio
    """
    if not distribution:
        return {}
    
    counts = list(distribution.values())
    
    stats = {
        'total_images': sum(counts),
        'num_classes': len(counts),
        'min_count': min(counts),
        'max_count': max(counts),
        'mean_count': np.mean(counts),
        'std_count': np.std(counts),
        'imbalance_ratio': max(counts) / min(counts) if min(counts) > 0 else float('inf'),
    }
    
    # Tìm class nhiều nhất và ít nhất
    sorted_items = sorted(distribution.items(), key=lambda x: x[1])
    stats['smallest_class'] = sorted_items[0]
    stats['largest_class'] = sorted_items[-1]
    
    return stats


def show_raw_samples(data_dir: Path, save_path: Path = None, samples_per_class: int = 3):
    """
    Hiển thị mẫu ảnh gốc từ mỗi class.
    
    Args:
        data_dir: Thư mục chứa các folder class
        save_path: Đường dẫn lưu hình
        samples_per_class: Số mẫu mỗi class
    """
    class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
    n_classes = len(class_dirs)
    
    if n_classes == 0:
        print("⚠️ Không tìm thấy class nào!")
        return
    
    fig, axes = plt.subplots(n_classes, samples_per_class, 
                             figsize=(samples_per_class * 3, n_classes * 2.5))
    
    for row, class_dir in enumerate(class_dirs):
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        samples = random.sample(images, min(samples_per_class, len(images)))
        
        for col in range(samples_per_class):
            ax = axes[row, col] if n_classes > 1 else axes[col]
            
            if col < len(samples):
                img = cv2.imread(str(samples[col]))
                if img is not None:
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    ax.imshow(img_rgb)
                    if col == 0:
                        ax.set_ylabel(class_dir.name, fontsize=8, rotation=0, 
                                     ha='right', va='center')
            ax.axis('off')
    
    plt.suptitle('🖼️ Raw Samples per Class (BEFORE Processing)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        if 'save_figure' in globals() and callable(save_figure):
            try:
                save_figure(fig, save_path.name, save_path.parent)
            except Exception:
                fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        else:
            fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"💾 Saved: {save_path}")
    
    plt.show()


# =============================================================================
# PHÂN TÍCH TẬP TRAIN (sử dụng actual_input_dir từ bước split)
# =============================================================================
print("=" * 60)
print("📊 EDA - PHÂN TÍCH DỮ LIỆU TRƯỚC XỬ LÝ")
print("=" * 60)

# Sử dụng actual_input_dir (đã được set ở cell split)
train_dir = actual_input_dir / 'train'
val_dir = actual_input_dir / 'val'
test_dir = actual_input_dir / 'test'

print(f"\n📁 Đang phân tích từ: {actual_input_dir}")

# Lấy phân phối
train_dist_before = get_class_distribution(train_dir)
val_dist = get_class_distribution(val_dir)
test_dist = get_class_distribution(test_dir)

if train_dist_before:
    print(f"\n📁 Tìm thấy {len(train_dist_before)} classes trong tập TRAIN")
    
    # Phân tích imbalance
    stats = analyze_imbalance(train_dist_before)
    
    print(f"\n📈 Thống kê TRAIN set:")
    print(f"   Tổng số ảnh: {stats['total_images']:,}")
    print(f"   Số classes: {stats['num_classes']}")
    print(f"   Min/Max: {stats['min_count']} / {stats['max_count']}")
    print(f"   Mean ± Std: {stats['mean_count']:.1f} ± {stats['std_count']:.1f}")
    print(f"   Imbalance Ratio: {stats['imbalance_ratio']:.2f}x")
    print(f"   Class ít nhất: {stats['smallest_class'][0]} ({stats['smallest_class'][1]} ảnh)")
    print(f"   Class nhiều nhất: {stats['largest_class'][0]} ({stats['largest_class'][1]} ảnh)")
    
    # Thống kê VAL và TEST
    if val_dist:
        val_total = sum(val_dist.values())
        print(f"\n📈 Thống kê VAL set: {val_total:,} ảnh ({len(val_dist)} classes)")
    if test_dist:
        test_total = sum(test_dist.values())
        print(f"📈 Thống kê TEST set: {test_total:,} ảnh ({len(test_dist)} classes)")
    
    # Cảnh báo imbalance
    if stats['imbalance_ratio'] > 3:
        print(f"\n⚠️  CẢNH BÁO: Dữ liệu MẤT CÂN BẰNG NGHIÊM TRỌNG!")
        print(f"   Imbalance ratio {stats['imbalance_ratio']:.1f}x > 3x")
        print(f"   → Sẽ thực hiện AUTO-BALANCING trong bước xử lý!")
    elif stats['imbalance_ratio'] > 1.5:
        print(f"\n⚠️  Dữ liệu có mất cân bằng nhẹ (ratio {stats['imbalance_ratio']:.1f}x)")
    else:
        print(f"\n✅ Dữ liệu tương đối cân bằng")
    
    # Vẽ và lưu biểu đồ phân phối TRƯỚC xử lý
    print("\n📊 Phân phối số lượng ảnh TRƯỚC xử lý:")
    plot_class_distribution(
        train_dist_before, 
        '📊 TRAIN Set Distribution (BEFORE Processing)', 
        color='coral',
        save_path=PIC_CLASS_DIST / 'class_distribution_before.png'
    )
    
    # Hiển thị và lưu mẫu ảnh gốc
    print("\n🖼️ Mẫu ảnh gốc từ mỗi class:")
    show_raw_samples(
        train_dir, 
        save_path=PIC_RAW_SAMPLES / 'raw_samples_per_class.png',
        samples_per_class=4
    )
else:
    print(f"❌ Không tìm thấy dữ liệu tại: {train_dir}")
    print("   Kiểm tra lại đường dẫn INPUT_DIR trong CONFIG hoặc chạy lại cell Split!")

## 4. 🔄 Main Processing Loop

**Pipeline xử lý chính gồm 3 bước:**

| Bước | Tên | Áp dụng cho | Mô tả |
|------|-----|-------------|-------|
| 1 | **Deduplication** | Train only | Loại bỏ ảnh trùng lặp bằng pHash |
| 2 | **Resize with Padding** | Tất cả | Chuẩn hóa kích thước, giữ tỉ lệ |
| 3 | **Auto-Balancing** | Train only | Augment để cân bằng số lượng mẫu |

> ⚠️ **Quan trọng:** Chỉ tập TRAIN được augment và balance. Tập VAL/TEST chỉ resize để đảm bảo đánh giá chính xác.

In [ ]:
# =============================================================================
# CELL 4: MAIN PROCESSING LOOP
# =============================================================================
# Pipeline: Deduplication → Resize → Balance
# VAL và TEST chỉ Resize, không augment
# =============================================================================

def process_dataset(input_dir: Path, output_dir: Path, target_size: Tuple[int, int],
                    balance: bool = True, augment: bool = True) -> Dict:
    """
    Xử lý toàn bộ dataset: Clean → Resize → Balance.
    
    Args:
        input_dir: Thư mục input chứa train/val/test
        output_dir: Thư mục output
        target_size: Kích thước đầu ra (width, height)
        balance: Có cân bằng số lượng mẫu không (chỉ cho train)
        augment: Có augment không (chỉ cho train)
        
    Returns:
        Dictionary thống kê kết quả
    """
    stats = {
        'train': {'original': 0, 'after_dedup': 0, 'final': 0, 'augmented': 0},
        'val': {'original': 0, 'final': 0},
        'test': {'original': 0, 'final': 0},
        'duplicates_removed': 0,
        'errors': 0
    }
    
    # Tạo augmentation pipeline
    transform = create_augmentation_pipeline() if augment else None
    
    # Xử lý từng split
    for split in ['train', 'val', 'test']:
        src_split_dir = input_dir / split
        dst_split_dir = output_dir / split
        
        if not src_split_dir.exists():
            print(f"⚠️  Không tìm thấy: {src_split_dir}")
            continue
        
        print(f"\n{'='*60}")
        print(f"🔄 Processing {split.upper()}...")
        print(f"{'='*60}")
        
        # Lấy danh sách classes
        class_dirs = [d for d in src_split_dir.iterdir() if d.is_dir()]
        
        # Xác định target count cho balancing (chỉ áp dụng cho train)
        target_count = 0
        if split == 'train' and balance:
            class_counts = {}
            for class_dir in class_dirs:
                images = list(class_dir.glob('*.jpg')) + \
                         list(class_dir.glob('*.jpeg')) + \
                         list(class_dir.glob('*.png'))
                class_counts[class_dir.name] = len(images)
            target_count = max(class_counts.values()) if class_counts else 0
            print(f"📊 Target count for balancing: {target_count}")
        
        # Xử lý từng class
        for class_dir in tqdm(class_dirs, desc=f'Processing {split} classes'):
            class_name = class_dir.name
            dst_class_dir = dst_split_dir / class_name
            dst_class_dir.mkdir(parents=True, exist_ok=True)
            
            # Lấy danh sách ảnh
            image_files = list(class_dir.glob('*.jpg')) + \
                          list(class_dir.glob('*.jpeg')) + \
                          list(class_dir.glob('*.png'))
            
            original_count = len(image_files)
            
            if split == 'train':
                stats['train']['original'] += original_count
            elif split == 'val':
                stats['val']['original'] += original_count
            else:
                stats['test']['original'] += original_count
            
            # =========================================================
            # BƯỚC 1: DEDUPLICATION (chỉ cho train)
            # =========================================================
            if split == 'train' and IMAGEHASH_AVAILABLE:
                duplicates = find_duplicates([str(p) for p in image_files], 
                                            threshold=CONFIG['phash_threshold'])
                duplicate_set = set(duplicates)
                image_files = [p for p in image_files if str(p) not in duplicate_set]
                stats['duplicates_removed'] += len(duplicates)
                stats['train']['after_dedup'] += len(image_files)
            
            # =========================================================
            # BƯỚC 2: RESIZE VÀ LƯU ẢNH GỐC
            # =========================================================
            saved_count = 0
            for img_path in image_files:
                img = load_image_safe(str(img_path))
                if img is None:
                    stats['errors'] += 1
                    continue
                
                # Resize với padding
                resized = resize_with_padding(img, target_size)
                
                # Lưu ảnh
                dst_path = dst_class_dir / img_path.name
                cv2.imwrite(str(dst_path), resized)
                saved_count += 1
            
            # =========================================================
            # BƯỚC 3: AUTO-BALANCING (chỉ cho train)
            # =========================================================
            if split == 'train' and balance and transform is not None:
                current_count = saved_count
                aug_count = 0
                
                # Augment cho đến khi đạt target_count
                while current_count < target_count and saved_count > 0:
                    # Chọn ngẫu nhiên một ảnh gốc để augment
                    src_img_path = random.choice(image_files)
                    img = load_image_safe(str(src_img_path))
                    
                    if img is None:
                        continue
                    
                    # Resize trước khi augment
                    resized = resize_with_padding(img, target_size)
                    
                    # Augment
                    augmented = augment_image(resized, transform)
                    
                    # Lưu với prefix aug_
                    aug_filename = f"aug_{aug_count}_{src_img_path.name}"
                    dst_path = dst_class_dir / aug_filename
                    cv2.imwrite(str(dst_path), augmented)
                    
                    current_count += 1
                    aug_count += 1
                
                stats['train']['augmented'] += aug_count
                stats['train']['final'] += current_count
            elif split == 'train':
                stats['train']['final'] += saved_count
            elif split == 'val':
                stats['val']['final'] += saved_count
            else:
                stats['test']['final'] += saved_count
    
    return stats


# =============================================================================
# CHẠY PROCESSING PIPELINE
# =============================================================================
print("=" * 60)
print("🔄 BẮT ĐẦU XỬ LÝ DATASET")
print("=" * 60)
print(f"📁 Input (splits): {actual_input_dir}")
PROCESSED_DIR = OUTPUT_DIR / 'processed'
print(f"📁 Processed Output: {PROCESSED_DIR}")
print(f"📐 Target Size: {TARGET_SIZE}")
print(f"⚖️  Auto-Balance: {CONFIG['balance_data']}")

# Xóa output cũ nếu có (chỉ xóa folder processed)
if PROCESSED_DIR.exists():
    print(f"\n⚠️  Xóa processed output cũ: {PROCESSED_DIR}")
    shutil.rmtree(PROCESSED_DIR)

# Chạy pipeline
processing_stats = process_dataset(
    actual_input_dir, 
    PROCESSED_DIR, 
    TARGET_SIZE,
    balance=CONFIG['balance_data'],
    augment=True
)

# In thống kê
print("\n" + "=" * 60)
print("✅ XỬ LÝ HOÀN TẤT!")
print("=" * 60)
print(f"\n📊 Thống kê TRAIN:")
print(f"   Ảnh gốc: {processing_stats['train']['original']:,}")
print(f"   Sau dedup: {processing_stats['train']['after_dedup']:,}")
print(f"   Ảnh augmented: {processing_stats['train']['augmented']:,}")
print(f"   Tổng final: {processing_stats['train']['final']:,}")

print(f"\n📊 Thống kê VAL:")
print(f"   Ảnh gốc: {processing_stats['val']['original']:,}")
print(f"   Final: {processing_stats['val']['final']:,}")

print(f"\n📊 Thống kê TEST:")
print(f"   Ảnh gốc: {processing_stats['test']['original']:,}")
print(f"   Final: {processing_stats['test']['final']:,}")

print(f"\n🗑️  Duplicates removed: {processing_stats['duplicates_removed']}")
print(f"❌ Errors: {processing_stats['errors']}")


## 5. 📊 EDA - Phân Tích Dữ Liệu SAU Xử Lý

**Mục đích:** Kiểm tra kết quả sau khi xử lý:
- Xác nhận phân phối đã cân bằng (imbalance ratio ≈ 1.0)
- Hiển thị mẫu ảnh ngẫu nhiên để kiểm tra chất lượng resize/augment
- Tổng kết số lượng ảnh cuối cùng

In [ ]:
# =============================================================================
# CELL 5: EDA - PHÂN TÍCH DỮ LIỆU SAU XỬ LÝ
# =============================================================================

def show_sample_grid(data_dir: Path, grid_size: Tuple[int, int] = (5, 5), 
                     title: str = "", save_path: Path = None):
    """
    Hiển thị grid ảnh ngẫu nhiên để kiểm tra chất lượng.

    Args:
        data_dir: Thư mục chứa các folder class
        grid_size: Kích thước grid (rows, cols)
        title: Tiêu đề
        save_path: Đường dẫn lưu hình (nếu None thì không lưu)
    
    Returns:
        Figure object
    """
    rows, cols = grid_size
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))

    # Thu thập tất cả ảnh
    all_images = []
    for class_dir in data_dir.iterdir():
        if class_dir.is_dir():
            for img_path in class_dir.glob('*.jpg'):
                all_images.append((img_path, class_dir.name))
            for img_path in class_dir.glob('*.png'):
                all_images.append((img_path, class_dir.name))

    if not all_images:
        print("⚠️ Không tìm thấy ảnh!")
        return None

    # Chọn ngẫu nhiên
    samples = random.sample(all_images, min(rows * cols, len(all_images)))

    for idx, (img_path, class_name) in enumerate(samples):
        row = idx // cols
        col = idx % cols

        img = cv2.imread(str(img_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img_rgb)
            axes[row, col].set_title(class_name, fontsize=8)
        axes[row, col].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Lưu hình nếu có save_path
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        if 'save_figure' in globals() and callable(save_figure):
            try:
                save_figure(fig, save_path.name, save_path.parent)
            except Exception:
                fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        else:
            fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"💾 Saved: {save_path}")
    
    plt.show()
    return fig


def show_samples_per_class(data_dir: Path, save_path: Path = None, samples_per_class: int = 4):
    """
    Hiển thị mẫu ảnh đã xử lý từ mỗi class.
    
    Args:
        data_dir: Thư mục chứa các folder class
        save_path: Đường dẫn lưu hình
        samples_per_class: Số mẫu mỗi class
    """
    class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
    n_classes = len(class_dirs)
    
    if n_classes == 0:
        print("⚠️ Không tìm thấy class nào!")
        return
    
    fig, axes = plt.subplots(n_classes, samples_per_class, 
                             figsize=(samples_per_class * 2, n_classes * 2))
    
    for row, class_dir in enumerate(class_dirs):
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        samples = random.sample(images, min(samples_per_class, len(images)))
        
        for col in range(samples_per_class):
            ax = axes[row, col] if n_classes > 1 else axes[col]
            
            if col < len(samples):
                img = cv2.imread(str(samples[col]))
                if img is not None:
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    ax.imshow(img_rgb)
                    if col == 0:
                        ax.set_ylabel(class_dir.name.split('_')[0], fontsize=9)
            ax.axis('off')
    
    plt.suptitle('🖼️ Processed Samples per Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        if 'save_figure' in globals() and callable(save_figure):
            try:
                save_figure(fig, save_path.name, save_path.parent)
            except Exception:
                fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        else:
            fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"💾 Saved: {save_path}")
    
    plt.show()


# =============================================================================
# PHÂN TÍCH SAU XỬ LÝ
# =============================================================================
print("=" * 60)
print("📊 EDA - PHÂN TÍCH DỮ LIỆU SAU XỬ LÝ")
print("=" * 60)

# Ensure PROCESSED_DIR exists in the namespace (fallback to OUTPUT_DIR/processed)
PROCESSED_DIR = globals().get('PROCESSED_DIR', OUTPUT_DIR / 'processed')

train_output_dir = PROCESSED_DIR / 'train'
val_output_dir = PROCESSED_DIR / 'val'
test_output_dir = PROCESSED_DIR / 'test'

# Lấy phân phối sau xử lý
train_dist_after = get_class_distribution(train_output_dir)
val_dist_after = get_class_distribution(val_output_dir)
test_dist_after = get_class_distribution(test_output_dir)

if train_dist_after:
    # Phân tích
    stats_after = analyze_imbalance(train_dist_after)

    print(f"\n📈 Thống kê TRAIN set SAU xử lý:")
    print(f"   Tổng số ảnh: {stats_after['total_images']:,}")
    print(f"   Số classes: {stats_after['num_classes']}")
    print(f"   Min/Max: {stats_after['min_count']} / {stats_after['max_count']}")
    print(f"   Imbalance Ratio: {stats_after['imbalance_ratio']:.2f}x")

    if stats_after['imbalance_ratio'] <= 1.1:
        print(f"\n✅ Dữ liệu đã được CÂN BẰNG hoàn hảo!")

    # Vẽ và lưu biểu đồ phân phối SAU xử lý
    print("\n📊 Phân phối số lượng ảnh SAU xử lý:")
    plot_class_distribution(
        train_dist_after, 
        '📊 TRAIN Set Distribution (AFTER Processing - BALANCED)', 
        color='seagreen',
        save_path=PIC_FINAL_DIST / 'class_distribution_after.png'
    )

    # Hiển thị và lưu grid mẫu ngẫu nhiên
    print("\n🖼️ Mẫu ảnh ngẫu nhiên sau xử lý:")
    show_sample_grid(
        train_output_dir, 
        (5, 5), 
        "🖼️ Random Samples from Processed TRAIN Set",
        save_path=PIC_SAMPLES / 'random_samples_processed.png'
    )
    
    # Hiển thị và lưu mẫu theo từng class
    print("\n🖼️ Mẫu ảnh theo từng class sau xử lý:")
    show_samples_per_class(
        train_output_dir,
        save_path=PIC_SAMPLES / 'samples_per_class_processed.png',
        samples_per_class=4
    )

# Thống kê tổng hợp
print("\n" + "=" * 60)
print("📋 TỔNG KẾT DATASET")
print("=" * 60)

total_train = sum(train_dist_after.values()) if train_dist_after else 0
total_val = sum(val_dist_after.values()) if val_dist_after else 0
total_test = sum(test_dist_after.values()) if test_dist_after else 0

print(f"\n📁 Processed Output Directory: {PROCESSED_DIR}")
print(f"   TRAIN: {total_train:,} images ({len(train_dist_after)} classes)")
print(f"   VAL:   {total_val:,} images ({len(val_dist_after)} classes)")
print(f"   TEST:  {total_test:,} images ({len(test_dist_after)} classes)")
print(f"   TOTAL: {total_train + total_val + total_test:,} images")

print(f"\n🖼️ Pictures saved to:")
print(f"   • {PIC_CLASS_DIST}")
print(f"   • {PIC_FINAL_DIST}")
print(f"   • {PIC_RAW_SAMPLES}")
print(f"   • {PIC_SAMPLES}")

## 🎉 Pipeline Complete!

---

### ✅ Các bước đã hoàn thành:

| # | Bước | Mô tả |
|---|------|-------|
| 1 | **Data Splitting** | Tự động chia 70/20/10 nếu chưa có splits |
| 2 | **Deduplication** | Loại bỏ ảnh trùng lặp bằng pHash |
| 3 | **Resize with Padding** | Chuẩn hóa 64×64, giữ nguyên tỷ lệ |
| 4 | **Auto-Balancing** | Augment để cân bằng số lượng mẫu giữa các class |
| 5 | **Export** | Tạo ZIP và metadata cho Kaggle |

---

### 📁 Cấu trúc Output:
```
output_CNNdataset/
├── train/
│   ├── 0_Cam_nguoc_chieu/
│   │   ├── image1.jpg
│   │   ├── aug_0_image1.jpg  (augmented)
│   │   └── ...
│   ├── 1_Cam_dung_va_do/
│   └── ...
├── val/
│   └── ... (chỉ resize, không augment)
├── test/
│   └── ... (chỉ resize, không augment)
└── dataset-metadata.json
```

---

### 🚀 Bước tiếp theo:
1. **Upload Kaggle**: Set `AUTO_UPLOAD = True` hoặc chạy lệnh thủ công
2. **Train Model**: Sử dụng dataset để train CNN model
3. **Evaluate**: Đánh giá trên tập test

---

### 📊 Tóm tắt kỹ thuật:
- **Input**: Ảnh biển báo giao thông Việt Nam (7 classes)
- **Output**: Dataset cân bằng, resize 64×64
- **Augmentation**: Rotate, Brightness, Blur, Noise (chỉ train)
- **Split Ratio**: 70% train / 20% val / 10% test